In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as pt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report


In [2]:
raw_file = 'training.csv'

In [3]:
raw_df = pd.read_csv(raw_file)

In [4]:
raw_df.head()

,EventId,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,...,PRI_jet_num,PRI_jet_leading_pt,PRI_jet_leading_eta,PRI_jet_leading_phi,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_subleading_phi,PRI_jet_all_pt,Weight,Label
0,100000,138.470,51.655,97.827,27.980,0.91,124.711,2.666,3.064,41.928,...,2,67.435,2.150,0.444,46.062,1.24,-2.475,113.497,0.002653,s
1,100001,160.937,68.768,103.235,48.146,-999.00,-999.000,-999.000,3.473,2.078,...,1,46.226,0.725,1.158,-999.000,-999.00,-999.000,46.226,2.233584,b
2,100002,-999.000,162.172,125.953,35.635,-999.00,-999.000,-999.000,3.148,9.336,...,1,44.251,2.053,-2.028,-999.000,-999.00,-999.000,44.251,2.347389,b
3,100003,143.905,81.417,80.943,0.414,-999.00,-999.000,-999.000,3.310,0.414,...,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,-0.000,5.446378,b
4,100004,175.864,16.915,134.805,16.405,-999.00,-999.000,-999.000,3.891,16.405,...,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,0.000,6.245333,b


In [5]:
raw_df.shape

(250000, 33)

In [6]:
raw_df.describe()

,EventId,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,...,PRI_met_sumet,PRI_jet_num,PRI_jet_leading_pt,PRI_jet_leading_eta,PRI_jet_leading_phi,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_subleading_phi,PRI_jet_all_pt,Weight
count,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,...,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000
mean,224999.500000,-49.023079,49.239819,81.181982,57.895962,-708.420675,-601.237051,-709.356603,2.373100,18.917332,...,209.797178,0.979176,-348.329567,-399.254314,-399.259788,-692.381204,-709.121609,-709.118631,73.064591,1.646767
std,72168.927986,406.345647,35.344886,40.828691,63.655682,454.480565,657.972302,453.019877,0.782911,22.273494,...,126.499506,0.977426,532.962789,489.338286,489.333883,479.875496,453.384624,453.389017,98.015662,1.875103
min,100000.000000,-999.000000,0.000000,6.329000,0.000000,-999.000000,-999.000000,-999.000000,0.208000,0.000000,...,13.678000,0.000000,-999.000000,-999.000000,-999.000000,-999.000000,-999.000000,-999.000000,0.000000,0.001502
25%,162499.750000,78.100750,19.241000,59.388750,14.068750,-999.000000,-999.000000,-999.000000,1.810000,2.841000,...,123.017500,0.000000,-999.000000,-999.000000,-999.000000,-999.000000,-999.000000,-999.000000,-0.000000,0.018636
50%,224999.500000,105.012000,46.524000,73.752000,38.467500,-999.000000,-999.000000,-999.000000,2.491500,12.315500,...,179.739000,1.000000,38.960000,-1.872000,-2.093000,-999.000000,-999.000000,-999.000000,40.512500,1.156188
75%,287499.250000,130.606250,73.598000,92.259000,79.169000,0.490000,83.446000,-4.593000,2.961000,27.591000,...,263.379250,2.000000,75.349000,0.433000,0.503000,33.703000,-2.457000,-2.275000,109.933750,2.404128
max,349999.000000,1192.026000,690.075000,1349.351000,2834.999000,8.503000,4974.979000,16.690000,5.684000,2834.999000,...,2003.976000,3.000000,1120.573000,4.499000,3.141000,721.456000,4.500000,3.142000,1633.433000,7.822543


In [7]:
print(raw_df['Label'].value_counts()) 

Label
b    164333
s     85667
Name: count, dtype: int64


In [8]:
print("\nTarget distribution:")
print(raw_df['Label'].value_counts())
print(raw_df['Label'].value_counts(normalize=True))


Target distribution:
Label
b    164333
s     85667
Name: count, dtype: int64
Label
b    0.657332
s    0.342668
Name: proportion, dtype: float64


In [9]:
raw_df = raw_df.replace(-999.0, np.nan)
print("\nMissing values per column (before handling):")
print(raw_df.isnull().sum()[raw_df.isnull().sum() > 0].sort_values(ascending=False))


Missing values per column (before handling):
DER_deltaeta_jet_jet      177457
DER_mass_jet_jet          177457
DER_prodeta_jet_jet       177457
PRI_jet_subleading_phi    177457
DER_lep_eta_centrality    177457
PRI_jet_subleading_eta    177457
PRI_jet_subleading_pt     177457
PRI_jet_leading_eta        99913
PRI_jet_leading_pt         99913
PRI_jet_leading_phi        99913
DER_mass_MMC               38114
dtype: int64


In [10]:
missing_cols = [
    'DER_deltaeta_jet_jet',
    'DER_mass_jet_jet',
    'DER_prodeta_jet_jet',
    'PRI_jet_subleading_phi',
    'DER_lep_eta_centrality',
    'PRI_jet_subleading_eta',
    'PRI_jet_subleading_pt',
    'PRI_jet_leading_eta',
    'PRI_jet_leading_pt',
    'PRI_jet_leading_phi',
    'DER_mass_MMC'
]


In [11]:
for col in missing_cols:
    raw_df[f'{col}_is_missing'] = raw_df[col].isnull().astype(int)

In [12]:
raw_df

,EventId,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,...,DER_mass_jet_jet_is_missing,DER_prodeta_jet_jet_is_missing,PRI_jet_subleading_phi_is_missing,DER_lep_eta_centrality_is_missing,PRI_jet_subleading_eta_is_missing,PRI_jet_subleading_pt_is_missing,PRI_jet_leading_eta_is_missing,PRI_jet_leading_pt_is_missing,PRI_jet_leading_phi_is_missing,DER_mass_MMC_is_missing
0,100000,138.470,51.655,97.827,27.980,0.91,124.711,2.666,3.064,41.928,...,0,0,0,0,0,0,0,0,0,0
1,100001,160.937,68.768,103.235,48.146,NaN,NaN,NaN,3.473,2.078,...,1,1,1,1,1,1,0,0,0,0
2,100002,NaN,162.172,125.953,35.635,NaN,NaN,NaN,3.148,9.336,...,1,1,1,1,1,1,0,0,0,1
3,100003,143.905,81.417,80.943,0.414,NaN,NaN,NaN,3.310,0.414,...,1,1,1,1,1,1,1,1,1,0
4,100004,175.864,16.915,134.805,16.405,NaN,NaN,NaN,3.891,16.405,...,1,1,1,1,1,1,1,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249995,349995,NaN,71.989,36.548,5.042,NaN,NaN,NaN,1.392,5.042,...,1,1,1,1,1,1,1,1,1,1
249996,349996,NaN,58.179,68.083,22.439,NaN,NaN,NaN,2.585,22.439,...,1,1,1,1,1,1,1,1,1,1
249997,349997,105.457,60.526,75.839,39.757,NaN,NaN,NaN,2.390,22.183,...,1,1,1,1,1,1,0,0,0,0
249998,349998,94.951,19.362,68.812,13.504,NaN,NaN,NaN,3.365,13.504,...,1,1,1,1,1,1,1,1,1,0


In [13]:
raw_df = raw_df.fillna(-999)
print("\nMissing values after handling:")
print(raw_df.isnull().sum().sum())


Missing values after handling:
0


In [14]:
X = raw_df.drop(['EventId', 'Label', 'Weight'], axis=1)
y = raw_df['Label'].map({'s': 1, 'b': 0})
weights = raw_df['Weight']

In [15]:
print(f"\nFeatures shape: {X.shape}")
print(f"Target distribution: {y.value_counts().to_dict()}")
print(f"Weight summary: min={weights.min():.2f}, max={weights.max():.2f}, mean={weights.mean():.2f}")


Features shape: (250000, 41)
Target distribution: {0: 164333, 1: 85667}
Weight summary: min=0.00, max=7.82, mean=1.65


In [16]:
X_train, X_val, y_train, y_val, w_train, w_val = train_test_split(
    X, y, weights, test_size=0.2, random_state=42, stratify=y
)

In [17]:
print(f"\nTrain size: {len(X_train)}, Validation size: {len(X_val)}")


Train size: 200000, Validation size: 50000


In [18]:
modelA = RandomForestClassifier(n_jobs=-1, random_state=42).fit(X_train, y_train)

In [19]:
y_predA = modelA.predict(X_val)
modelA.score(X_val, y_val)

0.8369

In [20]:
modelB = RandomForestClassifier(n_jobs=-1, random_state=42).fit(X_train, y_train, sample_weight=w_train)
y_predB = modelB.predict(X_val)

In [21]:
accuracy_score(y_val, y_predB)

0.82734

In [22]:
from sklearn.metrics import classification_report
print(classification_report(y_val, y_predA))  # Model A
print(classification_report(y_val, y_predB))  # Model B

              precision    recall  f1-score   support

           0       0.86      0.90      0.88     32867
           1       0.79      0.71      0.75     17133

    accuracy                           0.84     50000
   macro avg       0.82      0.81      0.81     50000
weighted avg       0.83      0.84      0.83     50000

              precision    recall  f1-score   support

           0       0.87      0.87      0.87     32867
           1       0.75      0.74      0.75     17133

    accuracy                           0.83     50000
   macro avg       0.81      0.81      0.81     50000
weighted avg       0.83      0.83      0.83     50000



In [23]:
def ams_score(y_true, y_pred, br=10):
    s = sum((y_true == 1) & (y_pred == 1))
    b = sum((y_true == 0) & (y_pred == 1))
    return np.sqrt(2 * ((s + b + br) * np.log(1 + s/(b + br)) - s))

print(ams_score(y_val, y_predA))  # Model A
print(ams_score(y_val, y_predB))  # Model B

154.67760686414925
146.98310035187677


In [24]:
def test_params(**params):
    model_alpha = RandomForestClassifier(n_jobs=-1, random_state=42, **params).fit(X_train, y_train, sample_weight=w_train)
    y_pred_alpha = model_alpha.predict(X_val)
    print(ams_score(y_val, y_pred_alpha))
    print(ams_score(y_val, y_predA))
    print(ams_score(y_val, y_predB))

In [25]:
test_params(max_depth=100)

146.83967643596895
154.67760686414925
146.98310035187677


In [26]:
test_params(max_depth=50)

140.28822699567655
154.67760686414925
146.98310035187677


In [27]:
test_params(max_leaf_nodes=2**20)

146.16962852562304
154.67760686414925
146.98310035187677


In [30]:
%%time
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Smaller parameter grid (8 combinations × 2 folds = 16 fits)
param_grid_small = {
    'n_estimators': [100, 200],
    'max_depth': [10, 15],
    'min_samples_split': [5]
}

# Use 2-fold CV instead of 3
grid_small = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid_small,
    cv=2,
    scoring=ams_score,
    n_jobs=-1,
    verbose=1
)

# Fit with weights (or without — your choice)
grid_small.fit(X_train, y_train, sample_weight=w_train)

C:\Users\shrey\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\model_selection\_search.py:883: UserWarning: The scoring <function ams_score at 0x0000027DD0FCE6C0> does not support sample_weight, which may lead to statistically incorrect results when fitting GridSearchCV(cv=2, estimator=RandomForestClassifier(n_jobs=-1, random_state=42),
             n_jobs=-1,
             param_grid={'max_depth': [10, 15], 'min_samples_split': [5],
                         'n_estimators': [100, 200]},
             scoring=<function ams_score at 0x0000027DD0FCE6C0>, verbose=1) with sample_weight. 
  warnings.warn(


Fitting 2 folds for each of 4 candidates, totalling 8 fits


C:\Users\shrey\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan]
  warnings.warn(


CPU times: total: 1min 45s
Wall time: 3min 33s


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [10, 15], 'min_samples_split': [5], 'n_estimators': [100, 200]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",<function ams...0027DD0FCE6C0>
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",2
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displa

In [33]:
# Final model with best parameters
final_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    n_jobs=-1,
    random_state=42
)

# Train on full training set (with weights)
final_model.fit(X_train, y_train, sample_weight=w_train)

# Evaluate on validation set
y_pred = final_model.predict(X_val)
print(f"Validation Accuracy: {accuracy_score(y_val, y_pred):.4f}")

ams_val = ams_score(y_val, y_pred)
print(f"Validation AMS: {ams_val:.4f}")

Validation Accuracy: 0.6630
Validation AMS: 37.0480


In [34]:
final_model_no_weight = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    n_jobs=-1,
    random_state=42
)
final_model_no_weight.fit(X_train, y_train)  # no sample_weight

y_pred_no_weight = final_model_no_weight.predict(X_val)
print(f"Accuracy (no weights): {accuracy_score(y_val, y_pred_no_weight):.4f}")
print(f"AMS (no weights): {ams_score(y_val, y_pred_no_weight):.4f}")

Accuracy (no weights): 0.8286
AMS (no weights): 149.3727


In [35]:
param_grid_small_157 = {
    'n_estimators': [100, 200],
    'min_samples_split': [2, 5],
    'max_depth': [None, 20],
    'max_features': ['sqrt', 0.5]
}
# 2 × 2 × 2 × 2 = 16 combinations × 2 folds = 32 fits → ~20 min

In [38]:
%%time
grid_157_small = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid_small_157,
    cv=2,
    scoring=ams_score,
    n_jobs=-1,
    verbose=1
)

grid_157_small.fit(X_train, y_train)

Fitting 2 folds for each of 16 candidates, totalling 32 fits


C:\Users\shrey\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(


CPU times: total: 4min 59s
Wall time: 43min 24s


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [None, 20], 'max_features': ['sqrt', 0.5], 'min_samples_split': [2, 5], 'n_estimators': [100, 200]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",<function ams...0027DD0FCE6C0>
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",2
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fo

In [41]:
# This is the best model from GridSearchCV
final_model = grid_157_small.best_estimator_

# Evaluate on validation set one last time
y_pred = final_model.predict(X_val)
ams_val = ams_score(y_val, y_pred)
print(f"Validation AMS: {ams_val}")

Validation AMS: 154.67760686414925


In [43]:
print(':)')
print(':)')
print(':)')

:)
:)
:)


In [44]:
import pandas as pd
import numpy as np

# Load test data
test_df = pd.read_csv('test.csv')

# Apply same preprocessing
test_df = test_df.replace(-999.0, np.nan)
for col in missing_cols:
    test_df[f'{col}_is_missing'] = test_df[col].isnull().astype(int)
test_df = test_df.fillna(-999)

# Get probabilities (not just predictions)
X_test = test_df.drop(['EventId'], axis=1)
y_proba = final_model.predict_proba(X_test)[:, 1]  # Probability of being signal (class 1)

# Create submission DataFrame
submission = pd.DataFrame({
    'EventId': test_df['EventId'],
    'Probability': y_proba
})

# Rank by probability (highest probability = rank 1)
submission['RankOrder'] = submission['Probability'].rank(ascending=False, method='first').astype(int)

# Map probability to class (threshold 0.5)
submission['Class'] = np.where(submission['Probability'] >= 0.5, 's', 'b')

# Keep only required columns
submission = submission[['EventId', 'RankOrder', 'Class']]

# Save
submission.to_csv('submission.csv', index=False)
print("✅ submission.csv created!")
print(submission.head(10))

✅ submission.csv created!
   EventId  RankOrder Class
0   350000     522005     b
1   350001     285221     b
2   350002     161518     s
3   350003      42286     s
4   350004     476477     b
5   350005     415313     b
6   350006     285222     b
7   350007     458344     b
8   350008     497260     b
9   350009      86278     s
